<a href="https://colab.research.google.com/github/Tl4826/Myproject/blob/feature%2Fmodel/ml/notebooks/01_train_food_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 - Fine-tune a dish classifier (RQ1)

Goal: recognise dishes from photos. Pretrained EfficientNet-B0 is fine-tuned on a small set of Food-101 classes, plus our own Vietnamese photos if present.
Run on Google Colab: Runtime > Change runtime type > GPU. Sources: Food-101 (Bossard et al., 2014), EfficientNet (Tan & Le, 2019).

In [ ]:
!pip -q install onnx onnxruntime scikit-learn

## 1. Config
Edit `CLASSES` (Food-101 folder names). Own photos: put them in `OWN_DIR/<class_name>/*.jpg`; they are added to the training split only, so the test split stays comparable.

In [2]:
import json, os, random, time
from pathlib import Path

import numpy as np
import torch, torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch.utils.data import ConcatDataset, DataLoader, Dataset, Subset
from torchvision import datasets, models, transforms

CLASSES = ["pho", "spring_rolls", "fried_rice", "ramen", "sushi",
           "pizza", "hamburger", "french_fries", "caesar_salad", "omelette"]
OWN_DIR = Path("/content/own_photos")      # optional Vietnamese photos
ROOT = Path("/content/data")               # Food-101 download (~5 GB, stays on Colab)
OUT = Path("/content/drive/MyDrive/food_ai") # checkpoints survive a disconnected session
BATCH, LR, SEED = 32, 1e-3, 42
EPOCHS = 1

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

device: cuda


Optional: mount Google Drive so checkpoints and the final model are not lost when Colab disconnects.

In [3]:
from google.colab import drive
drive.mount("/content/drive")
OUT.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


## 2. Data
Food-101 ships with an official train/test split. We keep only `CLASSES`, hold out 10% of train as validation, and check for leakage between splits.

In [ ]:
food = {s: datasets.Food101(ROOT, split=s, download=True) for s in ("train", "test")}
cls_idx = {c: i for i, c in enumerate(CLASSES)}
food_to_new = {food["train"].class_to_idx[c]: cls_idx[c] for c in CLASSES}

norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
tf_train = transforms.Compose([transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
                               transforms.RandomHorizontalFlip(), transforms.ToTensor(), norm])
tf_eval = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
                              transforms.ToTensor(), norm])

class Items(Dataset):
    # items: list of (image_path, new_label)
    def __init__(self, items, tf): self.items, self.tf = items, tf
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]
        return self.tf(Image.open(p).convert("RGB")), y

def food_items(split):
    ds = food[split]
    return [(str(ds._image_files[i]), food_to_new[l]) for i, l in enumerate(ds._labels) if l in food_to_new]

own = [(str(p), cls_idx[d.name]) for d in OWN_DIR.glob("*") if d.name in cls_idx
       for p in d.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}] if OWN_DIR.exists() else []

train_all = food_items("train")
random.shuffle(train_all)
n_val = len(train_all) // 10
val_items, train_items = train_all[:n_val], train_all[n_val:] + own
test_items = food_items("test")

# leakage check: no image path appears in two splits
paths = [set(p for p, _ in s) for s in (train_items, val_items, test_items)]
assert not (paths[0] & paths[1] or paths[0] & paths[2] or paths[1] & paths[2]), "split leakage"

print({k: len(v) for k, v in dict(train=train_items, val=val_items, test=test_items, own_added=own).items()})
print("images per class (train):", np.bincount([y for _, y in train_items], minlength=len(CLASSES)))

loaders = {name: DataLoader(Items(items, tf), batch_size=BATCH, shuffle=(name == "train"), num_workers=2)
           for name, items, tf in (("train", train_items, tf_train), ("val", val_items, tf_eval),
                                   ("test", test_items, tf_eval))}

 75%|███████▍  | 3.72G/5.00G [02:49<00:57, 22.1MB/s]

## 3. Model
EfficientNet-B0 pretrained on ImageNet: small enough for free Colab and CPU inference. Only the classifier head is trained first (fast, avoids ruining pretrained features), then the whole network is fine-tuned with a lower learning rate.

In [ ]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(CLASSES))
model.to(DEVICE)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

def run_epoch(loader, opt=None):
    model.train(opt is not None)
    tot, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(opt is not None):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x); loss = loss_fn(out, y)
            if opt: opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(y); correct += (out.argmax(1) == y).sum().item(); n += len(y)
    return tot / n, correct / n

def fit(params, lr, epochs, tag):
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    best = 0
    for e in range(epochs):
        t = time.time()
        tl, ta = run_epoch(loaders["train"], opt)
        vl, va = run_epoch(loaders["val"])
        print(f"[{tag}] epoch {e+1}/{epochs} train {tl:.3f}/{ta:.3f}  val {vl:.3f}/{va:.3f}  {time.time()-t:.0f}s")
        if va > best:
            best = va; torch.save(model.state_dict(), OUT / "best.pt")   # checkpoint on best val
    return best

for p in model.features.parameters(): p.requires_grad = False
fit(model.classifier.parameters(), LR, 2, "head")
for p in model.parameters(): p.requires_grad = True
best_val = fit(model.parameters(), LR / 10, EPOCHS, "full")
print("best val accuracy:", round(best_val, 4))

## 4. Evaluation (answers RQ1)
Evaluated once on the untouched test split.

In [ ]:
model.load_state_dict(torch.load(OUT / "best.pt", map_location=DEVICE)); model.eval()
ys, ps = [], []
with torch.no_grad():
    for x, y in loaders["test"]:
        ps += model(x.to(DEVICE)).argmax(1).cpu().tolist(); ys += y.tolist()
print("test accuracy:", round(accuracy_score(ys, ps), 4), " macro F1:", round(f1_score(ys, ps, average="macro"), 4))
print(classification_report(ys, ps, target_names=CLASSES, digits=3))

import matplotlib.pyplot as plt
cm = confusion_matrix(ys, ps)
plt.figure(figsize=(7, 6)); plt.imshow(cm, cmap="Blues")
plt.xticks(range(len(CLASSES)), CLASSES, rotation=90); plt.yticks(range(len(CLASSES)), CLASSES)
plt.xlabel("predicted"); plt.ylabel("true"); plt.colorbar(); plt.tight_layout()
plt.savefig(OUT / "confusion_matrix.png", dpi=150); plt.show()

## 5. Export for the web app
Exports ONNX (runs on CPU with `onnxruntime`, in Python or Node.js) plus `classes.json`. Preprocessing the web service must repeat: resize 256, center crop 224, scale to 0-1, normalise with mean (0.485, 0.456, 0.406) and std (0.229, 0.224, 0.225).

In [ ]:
import onnxruntime as ort
dummy = torch.randn(1, 3, 224, 224, device=DEVICE)
torch.onnx.export(model, dummy, OUT / "food_classifier.onnx", input_names=["image"], output_names=["logits"],
                  dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}}, opset_version=17)
json.dump({"classes": CLASSES}, open(OUT / "classes.json", "w"))

# self-check: ONNX output matches PyTorch
sess = ort.InferenceSession(str(OUT / "food_classifier.onnx"), providers=["CPUExecutionProvider"])
with torch.no_grad(): ref = model(dummy).cpu().numpy()
got = sess.run(None, {"image": dummy.cpu().numpy()})[0]
assert got.shape == (1, len(CLASSES)) and np.allclose(ref, got, atol=1e-3), "ONNX export mismatch"
print("export OK ->", OUT)